# 🎓 LectureScribe — Kaggle Edition

Transforms lecture videos (YouTube / direct URL) into structured, exam-ready PDF notes.

**Pipeline:** YouTube URL → yt-dlp + Deno → Whisper large-v3 (T4 GPU) → NVIDIA Nemotron-Ultra → Mermaid diagrams → ReportLab PDF

---
### Before running:
1. Add your NVIDIA API key: **Add-ons → Secrets → Add Secret** → name it `NVIDIA_API_KEY`
2. Enable **Internet** in notebook settings (right panel → Session options)
3. Enable **GPU T4 x2** accelerator
4. Fill in the **CONFIG cell** (Cell 3) and run all cells top to bottom

In [ ]:
# ── CELL 1: Environment Setup ─────────────────────────────────────────────────
# Installs all dependencies. Run once per session (~3-4 minutes).

import subprocess, sys, os

print('📦 Installing Python packages...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'openai-whisper', 'yt-dlp', 'openai',
    'python-dotenv', 'reportlab', 'rich',
    'Pillow', 'ffmpeg-python'
], check=True)
print('✅ Python packages installed')

# Install Deno — required by yt-dlp for YouTube downloads (avoids 403 errors)
print('📦 Installing Deno JS runtime (for yt-dlp YouTube support)...')
subprocess.run(
    'curl -fsSL https://deno.land/install.sh | sh',
    shell=True, capture_output=True
)
deno_bin = os.path.expanduser('~/.deno/bin')
os.environ['PATH'] = deno_bin + ':' + os.environ.get('PATH', '')
r = subprocess.run(['deno', '--version'], capture_output=True, text=True)
if r.returncode == 0:
    print(f'✅ Deno ready: {r.stdout.splitlines()[0]}')
else:
    print('⚠️  Deno not found — YouTube downloads may fail with 403')

# Install Mermaid CLI for diagram rendering
print('📦 Installing Mermaid CLI...')
subprocess.run(['npm', 'install', '-g', '@mermaid-js/mermaid-cli'], capture_output=True)
r = subprocess.run(['mmdc', '--version'], capture_output=True, text=True)
print(f'✅ mmdc {r.stdout.strip()} ready' if r.returncode == 0 else '⚠️  mmdc not found — diagrams will use placeholders')

# Check ffmpeg
r = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
print('✅ ffmpeg ready' if r.returncode == 0 else '❌ ffmpeg missing')

print('\n✅ Setup complete.')

In [ ]:
# ── CELL 2: Clone LectureScribe from GitHub ───────────────────────────────────

import os, sys

REPO_URL = 'https://github.com/vansh-kumar-007/lecturescribe.git'
REPO_DIR = '/kaggle/working/lecturescribe'

if os.path.exists(REPO_DIR):
    print('Repo already cloned. Pulling latest...')
    os.system(f'cd {REPO_DIR} && git pull')
else:
    print('Cloning repo...')
    os.system(f'git clone {REPO_URL} {REPO_DIR}')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.makedirs(f'{REPO_DIR}/outputs/diagrams', exist_ok=True)
os.makedirs(f'{REPO_DIR}/outputs/fonts', exist_ok=True)
os.chdir(REPO_DIR)

print(f'✅ LectureScribe ready at {REPO_DIR}')

In [ ]:
# ── CELL 3: CONFIG — Fill this in before running ──────────────────────────────

# Paste your YouTube or direct video URL here
VIDEO_URL = 'https://www.youtube.com/watch?v=XXXXXXXXXXX'

# Optional: override the lecture title in the PDF (leave '' to use Nemotron's title)
TITLE_OVERRIDE = ''

# Whisper model: 'large-v3' (best, fits on T4), 'medium' (faster)
WHISPER_MODEL = 'large-v3'

# Language hint for Whisper.
# For Hinglish (Hindi+English mixed): use 'hi'
# For English: use 'en'
# For auto-detect: use None
WHISPER_LANGUAGE = None  # None | 'en' | 'hi'

OUTPUT_DIR = f'{REPO_DIR}/outputs'

print('✅ Config set.')
print(f'   URL    : {VIDEO_URL}')
print(f'   Model  : Whisper {WHISPER_MODEL}')
print(f'   Lang   : {WHISPER_LANGUAGE or "auto-detect"}')

In [ ]:
# ── CELL 4: Load NVIDIA API Key from Kaggle Secrets ──────────────────────────
# Add your key: Add-ons → Secrets → Add Secret → name: NVIDIA_API_KEY

import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
NVIDIA_API_KEY = secrets.get_secret('NVIDIA_API_KEY')

# Write to .env so utils.get_env() picks it up
with open(f'{REPO_DIR}/.env', 'w') as f:
    f.write(f'NVIDIA_API_KEY={NVIDIA_API_KEY}\n')

print('✅ NVIDIA API key loaded from Kaggle Secrets.')

In [ ]:
# ── CELL 5: Step 1 — Download & Extract Audio ────────────────────────────────

import os, sys, time, subprocess
from pathlib import Path

# Re-add deno to PATH (Kaggle resets PATH between cells sometimes)
deno_bin = os.path.expanduser('~/.deno/bin')
if deno_bin not in os.environ.get('PATH', ''):
    os.environ['PATH'] = deno_bin + ':' + os.environ['PATH']

os.makedirs(OUTPUT_DIR, exist_ok=True)
output_wav = f'{OUTPUT_DIR}/audio.wav'

print('🎵 Downloading audio from YouTube...')
t0 = time.time()

dl = subprocess.run([
    'yt-dlp',
    '--js-runtime', 'deno',   # use deno to avoid 403 errors
    '-x',                      # extract audio only
    '--audio-format', 'wav',
    '--audio-quality', '0',
    '-o', f'{OUTPUT_DIR}/audio_raw.%(ext)s',
    VIDEO_URL
], capture_output=True, text=True)

if dl.returncode != 0:
    print('[ERROR] yt-dlp failed:')
    print(dl.stderr[-2000:])
    raise RuntimeError('Audio download failed. Check the URL and try again.')

# Find downloaded file
raw_files = list(Path(OUTPUT_DIR).glob('audio_raw*'))
if not raw_files:
    raise RuntimeError('yt-dlp finished but no audio file found in outputs/')
raw_file = str(raw_files[0])

# Resample to 16kHz mono WAV (required by Whisper)
print('  Resampling to 16kHz mono WAV...')
ff = subprocess.run([
    'ffmpeg', '-y', '-i', raw_file,
    '-ar', '16000', '-ac', '1',
    '-acodec', 'pcm_s16le',
    output_wav
], capture_output=True, text=True)

if ff.returncode != 0:
    raise RuntimeError(f'ffmpeg resample failed: {ff.stderr[-500:]}')

if raw_file != output_wav and os.path.exists(raw_file):
    os.remove(raw_file)

audio_path = output_wav
size_mb = os.path.getsize(audio_path) / 1024 / 1024
print(f'✅ Audio ready in {time.time()-t0:.1f}s')
print(f'   File : {audio_path}')
print(f'   Size : {size_mb:.1f} MB')

In [ ]:
# ── CELL 6: Step 2 — Transcribe with Whisper ─────────────────────────────────
# T4 GPU: ~1.5-2 hrs for a 12-hour video (vs ~6 hrs on RTX 3050)

import time, warnings
import torch
import whisper

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🎙 Transcribing with Whisper {WHISPER_MODEL} on {device.upper()}...')
if device == 'cpu':
    print('⚠️  No GPU detected! Enable GPU T4 in notebook settings for much faster transcription.')

t0 = time.time()

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    model = whisper.load_model(WHISPER_MODEL, device=device)

result = model.transcribe(
    audio_path,
    fp16=(device == 'cuda'),
    language=WHISPER_LANGUAGE,
    verbose=False
)

transcript = result['text'].strip()
word_count = len(transcript.split())
elapsed = time.time() - t0

# Save transcript
transcript_path = f'{OUTPUT_DIR}/transcript.txt'
with open(transcript_path, 'w', encoding='utf-8') as f:
    f.write(transcript)

print(f'✅ Transcription complete in {elapsed/60:.1f} min')
print(f'   Words    : ~{word_count}')
print(f'   Language : {result.get("language", "unknown")}')
print(f'   Saved to : {transcript_path}')
print(f'\nPreview (first 500 chars):')
print(transcript[:500])

In [ ]:
# ── CELL 7: (Optional) Load existing transcript — skip Cell 6 ────────────────
# Use this if you already transcribed and don't want to re-run Whisper.

import os
transcript_path = f'{OUTPUT_DIR}/transcript.txt'

if os.path.exists(transcript_path):
    with open(transcript_path, 'r', encoding='utf-8') as f:
        transcript = f.read()
    print(f'✅ Loaded existing transcript: ~{len(transcript.split())} words')
else:
    print('❌ No transcript found at', transcript_path)
    print('   Run Cell 6 first to transcribe.')

In [ ]:
# ── CELL 8: Step 3 — Analyze with NVIDIA Nemotron ───────────────────────────
# Chunks transcript and calls Nemotron-Ultra.
# Auto-retries on rate limits with 60s backoff.

import json, time, os
from collections import Counter

# Reload .env in case dotenv wasn't loaded yet
from dotenv import load_dotenv
load_dotenv(f'{REPO_DIR}/.env')

from nemotron import analyze_transcript

print('🧠 Analyzing transcript with Nemotron-Ultra...')
t0 = time.time()

notes = analyze_transcript(transcript)

if TITLE_OVERRIDE:
    notes['lecture_title'] = TITLE_OVERRIDE

notes_path = f'{OUTPUT_DIR}/notes.json'
with open(notes_path, 'w', encoding='utf-8') as f:
    json.dump(notes, f, indent=2)

block_types = [b['type'] for b in notes.get('blocks', [])]
counts = Counter(block_types)

print(f'✅ Analysis complete in {(time.time()-t0)/60:.1f} min')
print(f'   Title   : {notes.get("lecture_title")}')
print(f'   Subject : {notes.get("subject")}')
print(f'   Blocks  : {len(block_types)} total')
for btype, count in counts.items():
    print(f'     {btype:12s}: {count}')
print(f'   Saved to: {notes_path}')

In [ ]:
# ── CELL 9: (Optional) Load existing notes.json — skip Cell 8 ───────────────
# Use this if Nemotron already ran and you don't want to call it again.

import json, os
notes_path = f'{OUTPUT_DIR}/notes.json'

if os.path.exists(notes_path):
    with open(notes_path, 'r', encoding='utf-8') as f:
        notes = json.load(f)
    print(f'✅ Loaded existing notes: {len(notes.get("blocks", []))} blocks')
    print(f'   Title: {notes.get("lecture_title")}')
else:
    print('❌ No notes.json found at', notes_path)
    print('   Run Cell 8 first.')

In [ ]:
# ── CELL 10: Step 4 — Render Mermaid Diagrams ────────────────────────────────

import time, os
from diagram_renderer import render_diagrams

print('📊 Rendering Mermaid diagrams...')
t0 = time.time()

diagram_paths = render_diagrams(
    notes['blocks'],
    OUTPUT_DIR,
    notes.get('lecture_title', 'lecture')
)

print(f'✅ {len(diagram_paths)} diagram(s) rendered in {time.time()-t0:.1f}s')
for idx, path in diagram_paths.items():
    print(f'   Block {idx}: {os.path.basename(path)}')

In [ ]:
# ── CELL 11: Step 5 — Generate PDF ───────────────────────────────────────────

import time, os
from pdf_renderer import render_pdf

print('📄 Generating PDF...')
t0 = time.time()

pdf_path = render_pdf(notes, diagram_paths, OUTPUT_DIR)

size_kb = os.path.getsize(pdf_path) / 1024
print(f'✅ PDF generated in {time.time()-t0:.1f}s')
print(f'   Path : {pdf_path}')
print(f'   Size : {size_kb:.1f} KB')

In [ ]:
# ── CELL 12: Download Output Files ───────────────────────────────────────────
# Copies PDF, transcript, and notes.json to /kaggle/working/ for download.

import shutil, os
from pathlib import Path

files_to_copy = [
    (pdf_path, f'/kaggle/working/{Path(pdf_path).name}'),
    (f'{OUTPUT_DIR}/transcript.txt', '/kaggle/working/transcript.txt'),
    (f'{OUTPUT_DIR}/notes.json', '/kaggle/working/notes.json'),
]

for src, dst in files_to_copy:
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f'✅ {Path(dst).name} → ready to download')
    else:
        print(f'⚠️  {src} not found, skipping')

print()
print('📥 To download: Output panel (right side) → click the file → Download')

---
## 💡 Tips

**Hinglish videos:** Set `WHISPER_LANGUAGE = 'hi'` in Cell 3 for better accuracy on Hindi+English mixed content.

**Long videos (6-12 hrs):** Whisper on T4 takes ~1.5-2 hrs. Start the notebook and come back — Kaggle sessions stay alive up to 12 hours.

**Rate limits:** Nemotron allows ~32 requests/session. LectureScribe handles this automatically with 60s retry waits. A 12-hour video produces ~15-20 chunks — expect 15-30 min for Nemotron analysis.

**Resuming after interruption:**
- Transcript saved → skip Cell 6, run Cell 7 instead
- Nemotron done → skip Cell 8, run Cell 9 instead

**Local Udemy courses with SRTs:** Run LectureScribe locally — SRT mode is instant and needs no GPU.